# Attack Success Rate Evaluation Using BERTAttack

This notebook evaluates model robustness by measuring the success rate of adversarial attacks.

## Configuration
- **Attack**: BERTAttack (Li et al., 2020)
- **USE Threshold**: 0.8
- **Max Perturbation**: 10% of words
- **Query Budget**: 50 per example

In [1]:
# Imports
import os, gc, json, random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed
import textattack
from textattack.models.wrappers import HuggingFaceModelWrapper
from textattack.attack_recipes import BERTAttackLi2020
from textattack.constraints.overlap import MaxWordsPerturbed
from textattack.datasets import Dataset as TADataset
from textattack import Attacker
from textattack.attack_args import AttackArgs

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## 1. Load Dataset

In [2]:
# Load LIAR dataset
ds = load_dataset('ucsbnlp/liar', trust_remote_code=True)
print({k: len(ds[k]) for k in ds.keys()})

{'train': 10269, 'test': 1283, 'validation': 1284}


In [3]:
# Label configuration
LABELS = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

def safe_str(x):
    if x is None: return ''
    if isinstance(x, (list, tuple)):
        x = ', '.join([str(i) for i in x if i is not None])
    return str(x).strip()

def build_text(example):
    statement = safe_str(example.get('statement', ''))
    subject = safe_str(example.get('subjects', example.get('subject', '')))
    context = safe_str(example.get('context', ''))
    parts = [statement]
    if subject: parts.append(f'SUBJECT: {subject}')
    if context: parts.append(f'CONTEXT: {context}')
    example['text'] = ' [SEP] '.join(parts)
    y = example.get('label')
    if isinstance(y, str):
        example['label_id'] = label2id[y.strip().lower()]
    else:
        example['label_id'] = int(y)
    return example

ds = ds.map(build_text)
print(f"Example: {ds['test'][0]['text'][:150]}...")

Example: Building a wall on the U.S.-Mexico border will take literally years. [SEP] SUBJECT: immigration [SEP] CONTEXT: Radio interview...


## 2. Load Model

In [4]:
MODEL_DIR = 'baseline_42'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, model_max_length=MAX_LENGTH, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.config.id2label = id2label
model.config.label2id = label2id

print(f'Loaded: {MODEL_DIR} | num_labels: {model.config.num_labels}')

Loaded: baseline_42 | num_labels: 6


## 3. Configure Attack

In [5]:
# GPU setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
torch.set_grad_enabled(False)
print(f'Device: {device}')

# Build attack
model_wrapper = HuggingFaceModelWrapper(model, tokenizer)
attack = BERTAttackLi2020.build(model_wrapper)

# Configure constraints
USE_THRESHOLD = 0.8
MAX_PCT_WORDS = 0.10
MAX_CANDIDATES = 5


for c in attack.constraints:
    if c.__class__.__name__ == 'UniversalSentenceEncoder':
        c.threshold = USE_THRESHOLD
        print(f'USE threshold: {c.threshold}')

attack.transformation.max_length = MAX_LENGTH
if hasattr(attack.transformation, 'max_candidates'):
    attack.transformation.max_candidates = MAX_CANDIDATES

# Add word perturbation limit
attack.constraints = [c for c in attack.constraints 
                      if c.__class__.__name__ not in ('MaxPercentWordsPerturbed', 'MaxWordsPerturbed')]
attack.constraints.append(MaxWordsPerturbed(max_percent=MAX_PCT_WORDS))

print(attack)

Device: cuda


C:\Users\mailm\Desktop\Hamza\deBerta\textattack\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
text

USE threshold: 0.8
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapMaskedLM(
    (method):  bert-attack
    (masked_lm_name):  BertForMaskedLM
    (max_length):  128
    (max_candidates):  5
    (min_confidence):  0.0005
  )
  (constraints): 
    (0): UniversalSentenceEncoder(
        (metric):  cosine
        (threshold):  0.8
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (1): MaxWordsPerturbed(
        (max_percent):  0.1
        (compare_against_original):  True
      )
    (2): RepeatModification
    (3): StopwordModification
  (is_black_box):  True
)


## 4. Run Attack Evaluation

In [15]:
# Prepare test data
SPLIT = 'test'
N_EXAMPLES = 1283
split_ds = ds[SPLIT].select(range(min(N_EXAMPLES, len(ds[SPLIT]))))
ta_pairs = [(ex['text'], ex['label_id']) for ex in split_ds]
ta_pairs.pop(990)
ta_data = TADataset(ta_pairs)

print(f'Evaluating {len(ta_data)} examples from {SPLIT} split')

Evaluating 1282 examples from test split


In [16]:
# Setup output and checkpointing
OUT_DIR = 'ATTACK_SUCCESS_RATE'
os.makedirs(OUT_DIR, exist_ok=True)

STATE_PATH = os.path.join(OUT_DIR, f'progress_{SPLIT}.json')
JSONL_PATH = os.path.join(OUT_DIR, f'adv_success_{SPLIT}.jsonl')

def save_state(next_index, success_count):
    tmp = STATE_PATH + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump({'next_index': next_index, 'success_count': success_count}, f)
    os.replace(tmp, STATE_PATH)

def append_success(rows):
    with open(JSONL_PATH, 'a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Load progress
start_idx, success_count = 0, 0
if os.path.exists(STATE_PATH):
    with open(STATE_PATH, 'r', encoding='utf-8') as f:
        st = json.load(f)
    start_idx = int(st.get('next_index', 0))
    success_count = int(st.get('success_count', 0))
    print(f'Resuming from index {start_idx} (success: {success_count})')
else:
    print('Starting fresh evaluation')

Resuming from index 1271 (success: 922)


In [17]:
# Execute attack
BATCH_SIZE = 1283
total = len(ta_pairs)

print(f'\nStarting attack evaluation...')
print(f'Total: {total}, Batch size: {BATCH_SIZE}, Query budget: 80\n')

for i in range(start_idx, total, BATCH_SIZE):
    j = min(i + BATCH_SIZE, total)
    batch_pairs = ta_pairs[i:j]
    batch_data = TADataset(batch_pairs)
    
    attack_args = AttackArgs(
        num_examples=len(batch_data),
        query_budget=80,
        random_seed=SEED,
        disable_stdout=True,
        log_to_csv=None,
        
    )
    
    print(f'Processing batch [{i}:{j}]...')
    attacker = Attacker(attack, batch_data, attack_args)
    batch_results = attacker.attack_dataset()
    
    # Extract successful attacks
    rows = []
    for k, r in enumerate(batch_results):
        if r.__class__.__name__ == 'SuccessfulAttackResult':
            adv_text = r.perturbed_result.attacked_text.text
            label = int(r.original_result.ground_truth_output)
            rows.append({'text': adv_text, 'label': label, 'orig_index': i + k})
    
    success_count += len(rows)
    if rows:
        append_success(rows)
    save_state(next_index=j, success_count=success_count)
    
    batch_success_rate = (len(rows) / len(batch_data) * 100) if len(batch_data) > 0 else 0
    print(f'Batch: {len(rows)}/{len(batch_data)} ({batch_success_rate:.1f}%)')
    print(f'Total: {success_count}/{j} ({success_count/j*100:.1f}%)\n')
    
    del attacker, batch_results, batch_data
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n=== COMPLETE ===')
print(f'Total evaluated: {total}')
print(f'Successful attacks: {success_count}')
print(f'Attack Success Rate: {success_count/total*100:.2f}%')
print(f'\nResults: {JSONL_PATH}')


Starting attack evaluation...
Total: 1282, Batch size: 1283, Query budget: 80

Processing batch [1271:1282]...
Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  unk
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapMaskedLM(
    (method):  bert-attack
    (masked_lm_name):  BertForMaskedLM
    (max_length):  128
    (max_candidates):  5
    (min_confidence):  0.0005
  )
  (constraints): 
    (0): UniversalSentenceEncoder(
        (metric):  cosine
        (threshold):  0.8
        (window_size):  inf
        (skip_text_shorter_than_window):  False
        (compare_against_original):  True
      )
    (1): MaxWordsPerturbed(
        (max_percent):  0.1
        (compare_against_original):  True
      )
    (2): RepeatModification
    (3): StopwordModification
  (is_black_box):  True
) 




  9%|███████▌                                                                           | 1/11 [00:00<00:01,  5.03it/s]
[Succeeded / Failed / Skipped / Total] 0 / 0 / 1 / 1:   9%|██▋                          | 1/11 [00:00<00:01,  5.01it/s]
[Succeeded / Failed / Skipped / Total] 0 / 0 / 1 / 1:  18%|█████▎                       | 2/11 [00:00<00:03,  2.51it/s]
[Succeeded / Failed / Skipped / Total] 0 / 1 / 1 / 2:  18%|█████▎                       | 2/11 [00:00<00:03,  2.50it/s]
[Succeeded / Failed / Skipped / Total] 0 / 1 / 2 / 3:  27%|███████▉                     | 3/11 [00:00<00:02,  3.66it/s]
[Succeeded / Failed / Skipped / Total] 0 / 1 / 2 / 3:  36%|██████████▌                  | 4/11 [00:01<00:01,  3.91it/s]
[Succeeded / Failed / Skipped / Total] 1 / 1 / 2 / 4:  36%|██████████▌                  | 4/11 [00:01<00:01,  3.91it/s]
[Succeeded / Failed / Skipped / Total] 2 / 1 / 2 / 5:  45%|█████████████▏               | 5/11 [00:01<00:01,  3.68it/s]
[Succeeded / Failed / Skipped / Total] 


+-------------------------------+--------+
| Attack Results                |        |
+-------------------------------+--------+
| Number of successful attacks: | 2      |
| Number of failed attacks:     | 1      |
| Number of skipped attacks:    | 8      |
| Original accuracy:            | 27.27% |
| Accuracy under attack:        | 9.09%  |
| Attack success rate:          | 66.67% |
| Average perturbed word %:     | 4.79%  |
| Average num. words per input: | 26.45  |
| Avg num queries:              | 28.0   |
+-------------------------------+--------+


Batch: 2/11 (18.2%)
Total: 924/1282 (72.1%)


=== COMPLETE ===
Total evaluated: 1282
Successful attacks: 924
Attack Success Rate: 72.07%

Results: ATTACK_SUCCESS_RATE\adv_success_test.jsonl


## 5. Analyze Results

In [ ]:
# Load results
rows = []
if os.path.exists(JSONL_PATH):
    with open(JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

success_df = pd.DataFrame(rows)
print(f'Total successful attacks: {len(success_df)}')

if len(success_df) > 0:
    # Label distribution
    print('\nLabel distribution:')
    for label_id, count in success_df['label'].value_counts().sort_index().items():
        print(f'  {id2label[label_id]}: {count} ({count/len(success_df)*100:.1f}%)')
    
    # Calculate metrics
    asr = len(success_df) / len(ds[SPLIT]) * 100
    print(f'\nAttack Success Rate: {asr:.2f}%')
    print(f'Model Robustness: {100-asr:.2f}%')
    
    # Show samples
    print('\nSample adversarial examples:')
    display(success_df.head())